In [0]:
%sql
-- Summary statistics for the sales cube
SELECT 
  COUNT(*) AS total_records,
  COUNT(DISTINCT customer_id) AS total_customers,
  COUNT(DISTINCT product_id) AS total_products,
  COUNT(DISTINCT region_name) AS total_regions,
  MIN(full_date) AS earliest_date,
  MAX(full_date) AS latest_date,
  ROUND(SUM(revenue_usd), 2) AS total_revenue_usd,
  ROUND(SUM(profit_usd), 2) AS total_profit_usd,
  ROUND(AVG(discount_percentage), 2) AS avg_discount_pct
FROM gold_catalog.gold_schema.cube_sales_analytics

In [0]:
%sql
CREATE OR REPLACE TABLE gold_catalog.gold_schema.cube_sales_analytics AS
SELECT 
  -- Fact identifiers
  f.invoice_line_id,
  f.invoice_id,
  f.invoice_status,
  
  -- Date dimension
  d.full_date,
  d.year,
  d.month,
  d.day,
  d.weekday,
  
  -- Customer dimension
  c.customer_id,
  c.customer_name,
  c.customer_type,
  c.country AS customer_country,
  c.signup_date,
  
  -- Product dimension
  p.product_id,
  p.product_name,
  p.category,
  p.cost_price,
  
  -- Region dimension
  r.region_code,
  r.region_name,
  r.country AS region_country,
  
  -- Fact measures
  f.quantity,
  f.unit_price,
  f.discount,
  f.gross_amount,
  f.discount_amount,
  f.net_amount,
  f.currency,
  f.rate_to_usd,
  f.revenue_usd,
  
  -- Calculated metrics
  ROUND(f.revenue_usd - (f.quantity * p.cost_price * f.rate_to_usd), 2) AS profit_usd,
  CASE WHEN f.gross_amount > 0 THEN (f.discount_amount / f.gross_amount) * 100 ELSE 0 END AS discount_percentage
  
FROM gold_catalog.gold_schema.fact_sales f
LEFT JOIN gold_catalog.gold_schema.dim_date d 
  ON f.date_key = d.date_key
LEFT JOIN gold_catalog.gold_schema.dim_customer c 
  ON f.customer_key = c.customer_key
LEFT JOIN gold_catalog.gold_schema.dim_product p 
  ON f.product_key = p.product_key
LEFT JOIN gold_catalog.gold_schema.dim_region r 
  ON f.region_key = r.region_key

In [0]:
%sql
-- Preview the curated sales cube
SELECT 
  full_date,
  customer_name,
  product_name,
  category,
  region_name,
  quantity,
  net_amount,
  revenue_usd,
  profit_usd,
  discount_percentage
FROM gold_catalog.gold_schema.cube_sales_analytics
ORDER BY full_date DESC
LIMIT 10

In [0]:
%sql
-- Analyzing negative profit scenarios
SELECT 
  product_name,
  quantity,
  unit_price,
  cost_price,
  discount,
  discount_percentage,
  gross_amount,
  net_amount,
  revenue_usd,
  ROUND(quantity * cost_price * rate_to_usd, 2) AS total_cost_usd,
  profit_usd,
  CASE 
    WHEN profit_usd < 0 THEN 'Loss'
    WHEN profit_usd = 0 THEN 'Break-even'
    ELSE 'Profit'
  END AS profit_status
FROM gold_catalog.gold_schema.cube_sales_analytics
WHERE profit_usd < 0
ORDER BY profit_usd ASC
LIMIT 10

In [0]:
%sql
-- Analyzing profit distribution across discount levels
SELECT 
  CASE 
    WHEN discount_percentage = 0 THEN '0% (No discount)'
    WHEN discount_percentage <= 25 THEN '1-25%'
    WHEN discount_percentage <= 50 THEN '26-50%'
    WHEN discount_percentage <= 75 THEN '51-75%'
    ELSE '76-100%'
  END AS discount_range,
  COUNT(*) AS transaction_count,
  ROUND(AVG(profit_usd), 2) AS avg_profit_usd,
  ROUND(MIN(profit_usd), 2) AS min_profit_usd,
  ROUND(MAX(profit_usd), 2) AS max_profit_usd,
  SUM(CASE WHEN profit_usd < 0 THEN 1 ELSE 0 END) AS loss_transactions,
  ROUND(AVG(discount_percentage), 2) AS avg_discount_pct
FROM gold_catalog.gold_schema.cube_sales_analytics
GROUP BY 
  CASE 
    WHEN discount_percentage = 0 THEN '0% (No discount)'
    WHEN discount_percentage <= 25 THEN '1-25%'
    WHEN discount_percentage <= 50 THEN '26-50%'
    WHEN discount_percentage <= 75 THEN '51-75%'
    ELSE '76-100%'
  END
ORDER BY avg_discount_pct

In [0]:
%sql
select * from gold_catalog.gold_schema.cube_sales_analytics;